# 04. Fair Price Model

전처리 통합 데이터와 시차 분석 결과를 사용해 정책 미반영 전국 단위 주유소 적정가격과 적정 가격대를 산정합니다.

이 노트북은 Google Colab에서 단독 실행되도록 구성되어 있습니다. 설정 셀의 경로만 본인 Google Drive 구조에 맞게 수정한 뒤 위에서 아래로 실행하면 됩니다.


In [1]:
# ============================================================
# 04 공통 경로 설정
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os

ROOT_PATH = "/content/drive/MyDrive/Data_analysis/The appropriateness of domestic oil prices compared to international oil prices/산업부/"
PROCESSED_PATH = os.path.join(ROOT_PATH, "preprocessed_data") + "/"

print(f"ROOT_PATH      = {ROOT_PATH}")
print(f"PROCESSED_PATH = {PROCESSED_PATH}")
FAIR_PRICE_OUTPUT_PATH = os.path.join(ROOT_PATH, "적정가격대선정") + "/"
print(f"FAIR_PRICE_OUTPUT_PATH = {FAIR_PRICE_OUTPUT_PATH}")

# 필요한 패키지 확인/설치
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "tqdm": "tqdm",
    "IPython": "ipython",
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print(f"[설치] 필요한 패키지 설치: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
else:
    print("[확인] 필요한 패키지가 이미 설치되어 있습니다.")


Mounted at /content/drive
ROOT_PATH      = /content/drive/MyDrive/Data_analysis/The appropriateness of domestic oil prices compared to international oil prices/산업부/
PROCESSED_PATH = /content/drive/MyDrive/Data_analysis/The appropriateness of domestic oil prices compared to international oil prices/산업부/preprocessed_data/
FAIR_PRICE_OUTPUT_PATH = /content/drive/MyDrive/Data_analysis/The appropriateness of domestic oil prices compared to international oil prices/산업부/적정가격대선정/
[확인] 필요한 패키지가 이미 설치되어 있습니다.


In [2]:
# =========================================================
# 2단계-v2: 국제제품가격 기반 정유사/주유소 2층 적정가격 모델
# =========================================================

from __future__ import annotations

from pathlib import Path
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import HuberRegressor

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

try:
    from IPython.display import display
except Exception:
    display = None


# =========================================================
# 0. 사용자 설정
# =========================================================
DATE_COL = "date"

RAW_INPUT_FILE = Path(PROCESSED_PATH) / "분석용_일별_통합데이터.csv"

OUTPUT_DIR = Path(ROOT_PATH) / "적정가격대선정"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 실행할 유종: "gasoline" 또는 "diesel"
PRODUCT = "gasoline"

# rolling 학습 설정
DAILY_TRAIN_OBS_CAP = 900
DAILY_MIN_OBS = 240
DAILY_REFIT_EVERY = 7

WEEKLY_TRAIN_OBS_CAP = 260
WEEKLY_MIN_OBS = 80
WEEKLY_REFIT_EVERY = 4

# band 설정
BAND_SCORE_CAP = 1200
BAND_MIN_TOTAL = 120
BAND_REFIT_EVERY = 7
BAND_COVERAGE_GRID = [0.55, 0.60, 0.65, 0.70, 0.75, 0.80]
BAND_COVERAGE_MIN_FOR_SELECTION = 0.60

# 정책/충격 제외 구간
# 기존 2단계의 EXCLUDE_RANGES를 그대로 가져오되,
# 시작 전/종료 후 완충기간을 둔다.
EXCLUDE_BUFFER_BEFORE_DAYS = 14
EXCLUDE_BUFFER_AFTER_DAYS = 30

EXCLUDE_RANGES = [
    {"start": "2008-03-10", "end": "2008-12-31", "label": "fuel_tax_cut_2008"},
    {"start": "2011-04-07", "end": "2011-07-06", "label": "fuel_tax_cut_2011_100won"},
    {"start": "2018-11-06", "end": "2019-05-06", "label": "fuel_tax_cut_2018_15pct"},
    {"start": "2019-05-07", "end": "2019-08-31", "label": "fuel_tax_cut_2019_7pct"},
    {"start": "2021-11-12", "end": "2022-04-30", "label": "fuel_tax_cut_2021_20pct"},
    {"start": "2022-05-01", "end": "2022-06-30", "label": "fuel_tax_cut_2022_30pct"},
    {"start": "2022-07-01", "end": "2022-12-31", "label": "fuel_tax_cut_2022_37pct"},
    {"start": "2023-01-01", "end": "2024-06-30", "label": "fuel_tax_cut_2023_2024"},
    {"start": "2024-07-01", "end": "2024-10-31", "label": "fuel_tax_cut_2024_partial"},
    {"start": "2024-11-01", "end": "2025-04-30", "label": "fuel_tax_cut_2024_2025_partial"},
    {"start": "2025-05-01", "end": "2025-10-31", "label": "fuel_tax_cut_2025_readjusted"},
    {"start": "2025-11-01", "end": "2026-04-30", "label": "fuel_tax_cut_2025_2026_partial"},
    {"start": "2026-03-13", "end": "2026-03-26", "label": "price_cap_2026_round1"},
]

EXCLUDE_RANGES.append(
    {
        "start": "2026-01-01",
        "end": "2026-12-31",
        "label": "holdout_2026_full_year",
        "buffer_before_days": 0,
        "buffer_after_days": 0,
    }
)

# 제품별 컬럼 설정
# 경유는 기존 코드의 경유0.05_원리터가 아니라 경유0.001_원리터를 우선 사용한다.
PRODUCT_MAP = {
    "gasoline": {
        "label": "휘발유",
        "retail_col": "보통휘발유_평균",
        "intl_col_candidates": ["휘발유92RON_원리터", "휘발유95RON_원리터"],
        "refinery_pre_col_candidates": ["정유소_세전_보통휘발유"],
    },
    "diesel": {
        "label": "경유",
        "retail_col": "자동차용경유_평균",
        "intl_col_candidates": ["경유0.001_원리터", "경유0.05_원리터"],
        "refinery_pre_col_candidates": ["정유소_세전_자동차용경유"],
    },
}

# 2층 모델을 우선하되, 직접모델이 훨씬 좋으면 직접모델 선택
# two_layer_mae <= direct_mae * 1.15 이면 two_layer 선택
TWO_LAYER_MAE_ALLOWANCE = 1.15

# =========================================================
# 최종 모델 기준 설정
# =========================================================

# 최종 주유소 적정가격 모델에는 정유사 데이터를 절대 넣지 않는다.
USE_REFINERY_FOR_RETAIL_MODEL = False

# 정유사 모델은 별도 산출물/진단용으로만 실행한다.
RUN_REFINERY_DIAGNOSTIC = True

# 최종 주유소 모델은 direct로 고정한다.
FINAL_RETAIL_MODEL_MODE = "direct"

# 주유소 IRF는 당일 국제가격이 직접 들어가지 않도록 최소 1일 lag부터 사용한다.
MIN_CONSUMER_IRF_LAG_DAYS = 1

# 정유사 IRF는 별도 산출용이므로 주간 h=0 허용 가능
MIN_REFINERY_IRF_LAG_WEEKS = 0

print("[READY] 2단계-v2 설정 완료")

[READY] 2단계-v2 설정 완료


## 유틸 함수


In [3]:
# =========================================================
# 1. 유틸
# =========================================================

def print_log(msg: str):
    print(f"[STEP2-V2] {msg}")


def to_num(s):
    return pd.to_numeric(s, errors="coerce")


def safe_num_col(df: pd.DataFrame, col: str) -> pd.Series:
    if col in df.columns:
        return to_num(df[col])
    return pd.Series(np.nan, index=df.index, dtype=float)


def ensure_datetime_index(df: pd.DataFrame, date_col: str = DATE_COL) -> pd.DataFrame:
    out = df.copy()

    if isinstance(out.index, pd.DatetimeIndex):
        out.index = pd.to_datetime(out.index)
        out = out.sort_index()
        out.index.name = date_col
        return out

    if date_col not in out.columns:
        raise ValueError(f"{date_col} 컬럼이 없습니다.")

    out[date_col] = pd.to_datetime(out[date_col], errors="coerce")
    out = out.dropna(subset=[date_col])
    out = out.set_index(date_col).sort_index()
    out.index.name = date_col
    return out


def resolve_first_existing_col(df: pd.DataFrame, candidates: list[str], required: bool = True) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c

    if required:
        raise ValueError(f"필수 컬럼 후보가 하나도 없습니다: {candidates}")

    return None


def build_exclusion_mask(index: pd.DatetimeIndex,
                         exclude_ranges: list[dict],
                         before_days: int = EXCLUDE_BUFFER_BEFORE_DAYS,
                         after_days: int = EXCLUDE_BUFFER_AFTER_DAYS) -> pd.Series:
    idx = pd.to_datetime(index)
    mask = pd.Series(False, index=idx)

    for r in exclude_ranges:
        local_before = r.get("buffer_before_days", before_days)
        local_after = r.get("buffer_after_days", after_days)

        s = pd.to_datetime(r["start"]) - pd.Timedelta(days=local_before)
        e = pd.to_datetime(r["end"]) + pd.Timedelta(days=local_after)

        mask.loc[(idx >= s) & (idx <= e)] = True

    return mask


def add_calendar_features(index: pd.DatetimeIndex) -> pd.DataFrame:
    idx = pd.to_datetime(index)
    out = pd.DataFrame(index=idx)

    out["trend"] = np.arange(len(idx), dtype=float) / 365.25
    out["dow"] = idx.dayofweek
    out["month"] = idx.month
    out["quarter"] = idx.quarter

    wd = pd.get_dummies(
        pd.Series(idx.dayofweek, index=idx),
        prefix="wd",
        drop_first=True,
        dtype=float,
    )
    mm = pd.get_dummies(
        pd.Series(idx.month, index=idx),
        prefix="m",
        drop_first=True,
        dtype=float,
    )

    out = pd.concat([out, wd, mm], axis=1)
    out = out.drop(columns=["dow", "month", "quarter"], errors="ignore")
    return out


def robust_mad_scale(x) -> float:
    arr = np.asarray(x, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return np.nan

    med = np.median(arr)
    mad = np.median(np.abs(arr - med))
    scale = 1.4826 * mad

    if not np.isfinite(scale) or scale <= 1e-9:
        scale = np.std(arr)

    if not np.isfinite(scale) or scale <= 1e-9:
        scale = 1.0

    return float(scale)


def calc_acf(x, lag: int = 1) -> float:
    s = pd.Series(x).dropna().astype(float)
    if len(s) <= lag + 2:
        return np.nan
    return float(s.autocorr(lag=lag))


def load_stage1_irf(product: str, layer: str = "consumer"):
    """
    layer:
      - "consumer": 국제제품가격 -> 주유소 소비자가격 일별 시차
      - "refinery": 국제제품가격 -> 정유사 세전 공급가격 주간 시차

    최종 주유소 모델에는 consumer IRF만 사용한다.
    refinery IRF는 정유사 별도 산출용으로만 사용한다.
    """
    if layer == "consumer":
        lag_dir = Path(ROOT_PATH) / "시차분석" / f"{product}_lag_analysis"
        unit = "days"
        min_lag = MIN_CONSUMER_IRF_LAG_DAYS

    elif layer == "refinery":
        lag_dir = Path(ROOT_PATH) / "시차분석" / f"{product}_refinery_weekly_lag_analysis"
        unit = "weeks"
        min_lag = MIN_REFINERY_IRF_LAG_WEEKS

    else:
        raise ValueError("layer는 'consumer' 또는 'refinery' 여야 합니다.")

    summary_path = lag_dir / "analysis_summary.csv"
    irf_path = lag_dir / "impulse_response_path.csv"

    if not summary_path.exists():
        raise FileNotFoundError(f"1단계 summary 파일 없음: {summary_path}")

    if not irf_path.exists():
        raise FileNotFoundError(f"1단계 IRF 파일 없음: {irf_path}")

    summary = pd.read_csv(summary_path).iloc[0]
    irf = pd.read_csv(irf_path)

    use = irf[["h", "impulse_response"]].dropna().copy()
    use["h"] = use["h"].astype(int)
    use = use[use["h"] >= int(min_lag)].sort_values("h").reset_index(drop=True)

    if len(use) == 0:
        raise ValueError(
            f"{layer} IRF에서 h >= {min_lag} 조건을 만족하는 행이 없습니다: {irf_path}"
        )

    # 양의 반응만 우선 사용
    use["raw_weight"] = use["impulse_response"].clip(lower=0.0)

    # 양의 반응이 없으면 절대값 기준 fallback
    if float(use["raw_weight"].sum()) <= 0:
        use["raw_weight"] = use["impulse_response"].abs()

    use = use[use["raw_weight"] > 0].copy()

    if len(use) == 0:
        raise ValueError(f"IRF weight 생성 실패: {lag_dir}")

    use["weight"] = use["raw_weight"] / float(use["raw_weight"].sum())
    use["cum_weight"] = use["weight"].cumsum()
    use["layer"] = layer
    use["unit"] = unit

    print_log(f"[IRF LOAD] product={product}, layer={layer}, unit={unit}, min_lag={min_lag}")
    print_log(f"[IRF LOAD] source={lag_dir}")

    return summary, use

def weighted_lag_sum(series: pd.Series, weights: pd.DataFrame) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    pieces = []

    for _, row in weights.iterrows():
        h = int(row["h"])
        w = float(row["weight"])
        pieces.append(s.shift(h) * w)

    if len(pieces) == 0:
        return pd.Series(np.nan, index=series.index)

    return pd.concat(pieces, axis=1).sum(axis=1, min_count=1)


def make_irf_pressure_features(df: pd.DataFrame,
                               base_col: str,
                               weights: pd.DataFrame,
                               prefix: str) -> pd.DataFrame:
    out = df.copy()

    raw = pd.to_numeric(out[base_col], errors="coerce")
    raw_diff = raw.diff()

    # 1단계 IRF 기반 시차 반영 국제가격
    out[f"{prefix}_irf"] = weighted_lag_sum(raw, weights)

    # 상승/하락 압력도 IRF 가중치로 시차 반영
    out[f"{prefix}_irf_up"] = weighted_lag_sum(raw_diff.clip(lower=0.0), weights)
    out[f"{prefix}_irf_down"] = weighted_lag_sum(-raw_diff.clip(upper=0.0), weights)

    # 모델용 파생변수는 raw가 아니라 IRF series에서 만든다.
    irf = pd.to_numeric(out[f"{prefix}_irf"], errors="coerce")

    out[f"{prefix}_irf_diff_1"] = irf.diff()
    out[f"{prefix}_irf_diff_7"] = irf.diff(7)
    out[f"{prefix}_irf_vol_7"] = irf.diff().rolling(7, min_periods=3).std()
    out[f"{prefix}_irf_vol_14"] = irf.diff().rolling(14, min_periods=5).std()
    out[f"{prefix}_irf_ma_7"] = irf.rolling(7, min_periods=3).mean()
    out[f"{prefix}_irf_ma_30"] = irf.rolling(30, min_periods=10).mean()

    # raw 기반 변수는 진단용으로만 남긴다.
    # build_direct_retail_x_cols()에서 사용하지 않는다.
    out[f"{prefix}_raw_diff_1_diag"] = raw.diff()
    out[f"{prefix}_raw_diff_7_diag"] = raw.diff(7)
    out[f"{prefix}_raw_vol_7_diag"] = raw.diff().rolling(7, min_periods=3).std()

    return out


print("[READY] 유틸 함수 정의 완료")

[READY] 유틸 함수 정의 완료


## Rolling Huber 및 Band 함수


In [4]:
# =========================================================
# 2. rolling Huber + online band
# =========================================================

def scale_train_now(X_train: pd.DataFrame, X_now: pd.DataFrame):
    mu = X_train.mean(axis=0)
    sigma = X_train.std(axis=0, ddof=0).replace(0, 1.0)

    X_train_s = (X_train - mu) / sigma
    X_now_s = (X_now - mu) / sigma

    return X_train_s, X_now_s, mu, sigma


def rolling_huber_predict(
    data: pd.DataFrame,
    target_col: str,
    x_cols: list[str],
    *,
    exclude_col: str = "is_excluded",
    train_obs_cap: int = 900,
    min_obs: int = 240,
    refit_every: int = 7,
    show_progress: bool = True,
    desc: str = "rolling_huber",
) -> pd.DataFrame:
    tmp = data.copy().sort_index()

    for c in [target_col] + x_cols:
        tmp[c] = pd.to_numeric(tmp[c], errors="coerce")

    tmp[exclude_col] = tmp[exclude_col].fillna(1).astype(int)

    n = len(tmp)
    pred = np.full(n, np.nan)
    local_scale = np.full(n, np.nan)

    last_model = None
    last_mu = None
    last_sigma = None
    last_scale = np.nan
    last_refit_i = -10**9

    iterator = range(n)
    if show_progress:
        iterator = tqdm(iterator, total=n, desc=desc)

    for i in iterator:
        x_now = tmp.iloc[[i]][x_cols]

        if x_now.isna().any(axis=None):
            continue

        hist = tmp.iloc[:i].copy()
        hist = hist[hist[exclude_col].eq(0)]
        hist = hist.dropna(subset=[target_col] + x_cols)

        if len(hist) < min_obs:
            continue

        if len(hist) > train_obs_cap:
            hist = hist.iloc[-train_obs_cap:].copy()

        need_refit = (
            last_model is None
            or (i - last_refit_i >= refit_every)
        )

        if need_refit:
            X_train = hist[x_cols].astype(float)
            y_train = hist[target_col].astype(float)

            try:
                X_train_s, X_now_s, mu, sigma = scale_train_now(X_train, x_now.astype(float))

                model = HuberRegressor(
                    epsilon=1.35,
                    alpha=0.0001,
                    fit_intercept=True,
                    max_iter=1000,
                )
                model.fit(X_train_s, y_train)

                train_pred = model.predict(X_train_s)
                train_resid = y_train.to_numpy() - train_pred
                fit_scale = robust_mad_scale(train_resid)

                last_model = model
                last_mu = mu
                last_sigma = sigma
                last_scale = fit_scale
                last_refit_i = i

            except Exception:
                continue

        if last_model is None:
            continue

        try:
            X_now_s = (x_now.astype(float) - last_mu) / last_sigma
            pred[i] = float(last_model.predict(X_now_s)[0])
            local_scale[i] = last_scale
        except Exception:
            continue

    out = tmp.copy()
    out[f"pred_{target_col}"] = pred
    out[f"residual_{target_col}"] = out[target_col] - out[f"pred_{target_col}"]
    out[f"abs_residual_{target_col}"] = out[f"residual_{target_col}"].abs()
    out[f"local_scale_{target_col}"] = local_scale

    return out


def online_residual_band(
    data: pd.DataFrame,
    actual_col: str,
    pred_col: str,
    *,
    exclude_col: str = "is_excluded",
    coverage: float = 0.70,
    score_cap: int = BAND_SCORE_CAP,
    min_total: int = BAND_MIN_TOTAL,
    refit_every: int = BAND_REFIT_EVERY,
) -> pd.DataFrame:
    out = data.copy().sort_index()

    alpha = max(1.0 - float(coverage), 1e-6)
    low_q = alpha / 2.0
    high_q = 1.0 - alpha / 2.0

    band_low = np.full(len(out), np.nan)
    band_high = np.full(len(out), np.nan)
    hist_n_arr = np.full(len(out), np.nan)

    residual_hist = []
    last_low = np.nan
    last_high = np.nan
    last_refit_i = -10**9

    actual = pd.to_numeric(out[actual_col], errors="coerce")
    pred = pd.to_numeric(out[pred_col], errors="coerce")
    excluded = out[exclude_col].fillna(1).astype(int)

    for i in range(len(out)):
        hist_n_arr[i] = len(residual_hist)

        need_refit = (
            len(residual_hist) >= min_total
            and (not np.isfinite(last_low) or i - last_refit_i >= refit_every)
        )

        if need_refit:
            arr = np.asarray(residual_hist, dtype=float)
            arr = arr[np.isfinite(arr)]

            if len(arr) >= min_total:
                last_low = float(np.quantile(arr, low_q))
                last_high = float(np.quantile(arr, high_q))
                last_refit_i = i

        if np.isfinite(pred.iloc[i]) and np.isfinite(last_low) and np.isfinite(last_high):
            band_low[i] = pred.iloc[i] + last_low
            band_high[i] = pred.iloc[i] + last_high

        if (
            excluded.iloc[i] == 0
            and np.isfinite(actual.iloc[i])
            and np.isfinite(pred.iloc[i])
        ):
            residual_hist.append(float(actual.iloc[i] - pred.iloc[i]))
            if len(residual_hist) > score_cap:
                residual_hist = residual_hist[-score_cap:]

    out["band_low"] = band_low
    out["band_high"] = band_high
    out["band_hist_n"] = hist_n_arr
    out["band_coverage"] = float(coverage)

    valid = actual.notna() & out["band_low"].notna() & out["band_high"].notna()

    out["inside_band"] = np.where(
        valid,
        ((actual >= out["band_low"]) & (actual <= out["band_high"])).astype(int),
        np.nan,
    )
    out["below_band"] = np.where(
        valid,
        (actual < out["band_low"]).astype(int),
        np.nan,
    )
    out["above_band"] = np.where(
        valid,
        (actual > out["band_high"]).astype(int),
        np.nan,
    )

    judge = np.array([None] * len(out), dtype=object)
    judge[valid.to_numpy()] = "적정"
    judge[(valid & (actual < out["band_low"])).to_numpy()] = "저렴"
    judge[(valid & (actual > out["band_high"])).to_numpy()] = "비쌈"
    out["judge"] = judge

    return out


def interval_score(y: pd.Series, low: pd.Series, high: pd.Series, coverage: float) -> float:
    y = pd.to_numeric(y, errors="coerce")
    low = pd.to_numeric(low, errors="coerce")
    high = pd.to_numeric(high, errors="coerce")

    valid = y.notna() & low.notna() & high.notna()
    if not valid.any():
        return np.nan

    yv = y[valid]
    lv = low[valid]
    hv = high[valid]

    alpha = max(1.0 - float(coverage), 1e-6)
    score = hv - lv

    below = yv < lv
    above = yv > hv

    score.loc[below] = score.loc[below] + (2.0 / alpha) * (lv.loc[below] - yv.loc[below])
    score.loc[above] = score.loc[above] + (2.0 / alpha) * (yv.loc[above] - hv.loc[above])

    return float(score.mean())


def evaluate_prediction(
    df: pd.DataFrame,
    actual_col: str,
    pred_col: str,
    *,
    exclude_col: str = "is_excluded",
    label: str = "",
) -> pd.Series:
    z = df.copy()
    clean = z[z[exclude_col].fillna(1).eq(0)].copy()

    valid = clean[actual_col].notna() & clean[pred_col].notna()
    clean = clean[valid].copy()

    if len(clean) == 0:
        return pd.Series({
            "model_name": label,
            "clean_n": 0,
            "clean_mae": np.nan,
            "clean_median_ae": np.nan,
            "clean_bias": np.nan,
            "clean_abs_bias": np.nan,
            "clean_resid_acf1": np.nan,
            "clean_resid_acf7": np.nan,
        })

    resid = clean[actual_col] - clean[pred_col]

    return pd.Series({
        "model_name": label,
        "clean_n": int(len(clean)),
        "clean_mae": float(resid.abs().mean()),
        "clean_median_ae": float(resid.abs().median()),
        "clean_bias": float(resid.mean()),
        "clean_abs_bias": float(abs(resid.mean())),
        "clean_resid_acf1": float(abs(calc_acf(resid, 1))),
        "clean_resid_acf7": float(abs(calc_acf(resid, 7))),
    })


def select_best_band(
    df: pd.DataFrame,
    actual_col: str,
    pred_col: str,
    *,
    exclude_col: str = "is_excluded",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    banded_by_cov = {}

    for cov in BAND_COVERAGE_GRID:
        banded = online_residual_band(
            df,
            actual_col=actual_col,
            pred_col=pred_col,
            exclude_col=exclude_col,
            coverage=cov,
        )

        clean = banded[banded[exclude_col].fillna(1).eq(0)].copy()
        valid = clean[actual_col].notna() & clean["band_low"].notna() & clean["band_high"].notna()
        clean = clean[valid].copy()

        if len(clean) == 0:
            empirical_cov = np.nan
            width = np.nan
            score = np.nan
        else:
            empirical_cov = float(clean["inside_band"].mean())
            width = float((clean["band_high"] - clean["band_low"]).mean())
            score = interval_score(clean[actual_col], clean["band_low"], clean["band_high"], cov)

        rows.append({
            "coverage_target": cov,
            "clean_n": int(len(clean)),
            "clean_empirical_coverage": empirical_cov,
            "clean_mean_width": width,
            "clean_interval_score": score,
        })

        banded_by_cov[cov] = banded

    metrics = pd.DataFrame(rows)

    candidates = metrics[
        metrics["clean_empirical_coverage"] >= BAND_COVERAGE_MIN_FOR_SELECTION
    ].copy()

    if len(candidates) == 0:
        candidates = metrics.copy()

    chosen_cov = (
        candidates
        .sort_values(["clean_interval_score", "clean_mean_width"], na_position="last")
        .iloc[0]["coverage_target"]
    )

    return banded_by_cov[float(chosen_cov)], metrics


print("[READY] rolling model / band 함수 정의 완료")

[READY] rolling model / band 함수 정의 완료


## 분석 Frame 생성 함수


In [5]:
# =========================================================
# 3. 정유사/주유소 frame 생성
# =========================================================

def build_daily_retail_frame(raw_df: pd.DataFrame,
                             product: str,
                             weights: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    cfg = PRODUCT_MAP[product]

    df = ensure_datetime_index(raw_df, DATE_COL)

    retail_col = cfg["retail_col"]
    intl_col = resolve_first_existing_col(df, cfg["intl_col_candidates"], required=True)
    refinery_pre_col = resolve_first_existing_col(df, cfg["refinery_pre_col_candidates"], required=False)

    out = pd.DataFrame(index=df.index)
    out["actual_retail"] = safe_num_col(df, retail_col)
    out["intl_raw"] = safe_num_col(df, intl_col)

    if refinery_pre_col is not None:
        out["actual_refinery_pre_raw"] = safe_num_col(df, refinery_pre_col)
    else:
        out["actual_refinery_pre_raw"] = np.nan

    out["is_excluded"] = build_exclusion_mask(out.index, EXCLUDE_RANGES).astype(int)

    out = make_irf_pressure_features(out, "intl_raw", weights, prefix="intl")

    cal = add_calendar_features(out.index)
    out = out.join(cal, how="left")

    # 기존 데이터에 공휴일 플래그가 있으면 사용
    for c in ["is_holiday", "is_holiday_lag1", "is_holiday_lead1"]:
        if c in df.columns:
            out[c] = safe_num_col(df, c).fillna(0.0)

    meta = {
        "retail_col": retail_col,
        "intl_col": intl_col,
        "refinery_pre_col": refinery_pre_col,
        "label": cfg["label"],
    }

    return out, meta


FIRST_REFINERY_WEEK_END = "2008-05-10"


def rebuild_refinery_weekly_from_order_step2(
    daily_frame: pd.DataFrame,
    value_col: str = "actual_refinery_pre_raw",
    first_week_end: str = FIRST_REFINERY_WEEK_END,
) -> pd.DataFrame:
    tmp = (
        daily_frame.loc[daily_frame[value_col].notna(), [value_col]]
        .copy()
        .sort_index()
        .reset_index()
    )

    if len(tmp) == 0:
        return pd.DataFrame(columns=["week_end", "actual_refinery_pre", "source_date_in_csv"])

    week_index = pd.date_range(
        start=pd.Timestamp(first_week_end),
        periods=len(tmp),
        freq="W-SAT",
    )

    out = pd.DataFrame({
        "week_end": week_index,
        "actual_refinery_pre": pd.to_numeric(tmp[value_col], errors="coerce").values,
        "source_date_in_csv": tmp[DATE_COL].values if DATE_COL in tmp.columns else tmp.iloc[:, 0].values,
    })

    return out


def build_weekly_refinery_frame(
    daily_frame: pd.DataFrame,
    refinery_weights: pd.DataFrame | None = None,
) -> pd.DataFrame:
    d = daily_frame.copy().sort_index()

    if d["actual_refinery_pre_raw"].notna().sum() == 0:
        return pd.DataFrame()

    # 국제제품가격은 실제 일별 날짜 기준 주간 평균
    intl_weekly = pd.DataFrame()
    intl_weekly["intl_w"] = d["intl_raw"].resample("W-SAT").mean()
    intl_weekly["is_excluded"] = d["is_excluded"].resample("W-SAT").max()

    # 정유사 세전 공급가격은 1단계와 동일하게 순서 기반 주간 재구성
    refinery_weekly = rebuild_refinery_weekly_from_order_step2(
        d,
        value_col="actual_refinery_pre_raw",
        first_week_end=FIRST_REFINERY_WEEK_END,
    )

    if refinery_weekly.empty:
        return pd.DataFrame()

    refinery_weekly = refinery_weekly.set_index("week_end").sort_index()

    weekly = refinery_weekly.join(intl_weekly, how="inner")
    weekly = weekly.dropna(subset=["actual_refinery_pre", "intl_w"]).copy()

    # -----------------------------------------------------
    # 정유사 전용 1단계 IRF 반영
    # -----------------------------------------------------
    if refinery_weights is not None and len(refinery_weights) > 0:
        weekly["intl_w_irf"] = weighted_lag_sum(weekly["intl_w"], refinery_weights)

        diff = weekly["intl_w"].diff()
        weekly["intl_w_irf_up"] = weighted_lag_sum(diff.clip(lower=0.0), refinery_weights)
        weekly["intl_w_irf_down"] = weighted_lag_sum(-diff.clip(upper=0.0), refinery_weights)

    else:
        # fallback: 정유사 시차 결과가 없을 때만 사용
        weekly["intl_w_irf"] = weekly["intl_w"]
        weekly["intl_w_irf_up"] = weekly["intl_w"].diff().clip(lower=0.0)
        weekly["intl_w_irf_down"] = -weekly["intl_w"].diff().clip(upper=0.0)

    weekly["intl_w_diff"] = weekly["intl_w"].diff()
    weekly["intl_w_up"] = weekly["intl_w_diff"].clip(lower=0.0)
    weekly["intl_w_down"] = -weekly["intl_w_diff"].clip(upper=0.0)
    weekly["intl_w_vol4"] = weekly["intl_w_diff"].rolling(4, min_periods=2).std()
    weekly["intl_w_vol8"] = weekly["intl_w_diff"].rolling(8, min_periods=3).std()

    cal = add_calendar_features(weekly.index)
    weekly = weekly.join(cal, how="left")

    weekly.index.name = "week_end"
    return weekly

def week_end_sat_for_daily_index(index: pd.DatetimeIndex) -> pd.Series:
    idx = pd.to_datetime(index)
    return pd.Series(idx.to_period("W-SAT").end_time.normalize(), index=idx)


def attach_weekly_refinery_prediction(daily_frame: pd.DataFrame,
                                      weekly_pred: pd.DataFrame) -> pd.DataFrame:
    out = daily_frame.copy().sort_index()

    if weekly_pred is None or len(weekly_pred) == 0:
        out["week_end"] = week_end_sat_for_daily_index(out.index)
        out["fair_refinery_pre"] = np.nan
        out["refinery_band_low"] = np.nan
        out["refinery_band_high"] = np.nan
        return out

    w = weekly_pred.copy().sort_index()
    if "week_end" in w.columns:
        w = w.set_index("week_end")
    w.index = pd.to_datetime(w.index)

    out["week_end"] = week_end_sat_for_daily_index(out.index)

    pred_map = w["fair_refinery_pre"].to_dict()
    low_map = w["refinery_band_low"].to_dict() if "refinery_band_low" in w.columns else {}
    high_map = w["refinery_band_high"].to_dict() if "refinery_band_high" in w.columns else {}

    out["fair_refinery_pre"] = out["week_end"].map(pred_map)
    out["refinery_band_low"] = out["week_end"].map(low_map)
    out["refinery_band_high"] = out["week_end"].map(high_map)

    return out


def build_direct_retail_x_cols(daily_frame: pd.DataFrame) -> list[str]:
    """
    최종 주유소 direct 모델 feature.

    중요:
    - raw 당일 국제가격 변수는 사용하지 않는다.
    - 1단계에서 추정한 주유소 IRF를 적용한 변수만 사용한다.
    """
    base_cols = [
        "intl_irf",
        "intl_irf_up",
        "intl_irf_down",
        "intl_irf_diff_1",
        "intl_irf_diff_7",
        "intl_irf_vol_7",
        "intl_irf_vol_14",
        "intl_irf_ma_7",
        "intl_irf_ma_30",
        "trend",
    ]

    dummy_cols = [
        c for c in daily_frame.columns
        if c.startswith("wd_") or c.startswith("m_")
    ]

    holiday_cols = [
        c for c in ["is_holiday", "is_holiday_lag1", "is_holiday_lead1"]
        if c in daily_frame.columns
    ]

    x_cols = [
        c for c in base_cols + dummy_cols + holiday_cols
        if c in daily_frame.columns
    ]

    forbidden = [
        c for c in x_cols
        if c.startswith("intl_raw")
        or c in ["intl_diff_1", "intl_diff_7", "intl_vol_7", "intl_vol_14", "intl_ma_7", "intl_ma_30"]
    ]

    if forbidden:
        raise ValueError(f"주유소 direct 모델에 raw 국제가격 변수가 들어갔습니다: {forbidden}")

    return x_cols


def build_spread_x_cols(daily_frame: pd.DataFrame) -> list[str]:
    base_cols = [
        "fair_refinery_pre",
        "intl_irf_up",
        "intl_irf_down",
        "intl_diff_1",
        "intl_diff_7",
        "intl_vol_7",
        "intl_vol_14",
        "trend",
    ]

    dummy_cols = [c for c in daily_frame.columns if c.startswith("wd_") or c.startswith("m_")]
    holiday_cols = [c for c in ["is_holiday", "is_holiday_lag1", "is_holiday_lead1"] if c in daily_frame.columns]

    return [c for c in base_cols + dummy_cols + holiday_cols if c in daily_frame.columns]

def build_weekly_refinery_x_cols(weekly_frame: pd.DataFrame) -> list[str]:
    base_cols = [
        "intl_w_irf",
        "intl_w_irf_up",
        "intl_w_irf_down",
        "intl_w_diff",
        "intl_w_up",
        "intl_w_down",
        "intl_w_vol4",
        "intl_w_vol8",
        "trend",
    ]

    dummy_cols = [c for c in weekly_frame.columns if c.startswith("m_")]

    return [c for c in base_cols + dummy_cols if c in weekly_frame.columns]


print("[READY] frame 생성 함수 정의 완료")

[READY] frame 생성 함수 정의 완료


## 모델 실행 및 저장 함수


In [6]:
# =========================================================
# 4. 2단계-v2 메인 실행 함수
# =========================================================

def fit_refinery_weekly_model(weekly_frame: pd.DataFrame,
                              product: str) -> tuple[pd.DataFrame, pd.Series]:
    label = PRODUCT_MAP[product]["label"]

    if weekly_frame is None or len(weekly_frame) == 0:
        metrics = pd.Series({
            "model_name": f"{product}_refinery_weekly_huber",
            "status": "no_refinery_data",
            "clean_n": 0,
            "clean_mae": np.nan,
        })
        return pd.DataFrame(), metrics

    x_cols = build_weekly_refinery_x_cols(weekly_frame)

    fit = rolling_huber_predict(
        weekly_frame,
        target_col="actual_refinery_pre",
        x_cols=x_cols,
        exclude_col="is_excluded",
        train_obs_cap=WEEKLY_TRAIN_OBS_CAP,
        min_obs=WEEKLY_MIN_OBS,
        refit_every=WEEKLY_REFIT_EVERY,
        show_progress=True,
        desc=f"{label} 정유사 주간모델",
    )

    fit["fair_refinery_pre"] = fit["pred_actual_refinery_pre"]

    # 정유사 단계 band
    ref_banded, ref_band_metrics = select_best_band(
        fit.rename(columns={"fair_refinery_pre": "pred_refinery_tmp"}),
        actual_col="actual_refinery_pre",
        pred_col="pred_refinery_tmp",
        exclude_col="is_excluded",
    )

    fit["refinery_band_low"] = ref_banded["band_low"]
    fit["refinery_band_high"] = ref_banded["band_high"]
    fit["refinery_judge"] = ref_banded["judge"]

    fit["refinery_gap_원L"] = fit["actual_refinery_pre"] - fit["fair_refinery_pre"]

    metrics = evaluate_prediction(
        fit,
        actual_col="actual_refinery_pre",
        pred_col="fair_refinery_pre",
        exclude_col="is_excluded",
        label=f"{product}_refinery_weekly_huber",
    )
    metrics["status"] = "ok"
    metrics["x_cols"] = "|".join(x_cols)

    fit.index.name = "week_end"
    return fit, metrics


def fit_retail_direct_model(daily_frame: pd.DataFrame,
                            product: str) -> tuple[pd.DataFrame, pd.Series]:
    label = PRODUCT_MAP[product]["label"]

    x_cols = build_direct_retail_x_cols(daily_frame)

    fit = rolling_huber_predict(
        daily_frame,
        target_col="actual_retail",
        x_cols=x_cols,
        exclude_col="is_excluded",
        train_obs_cap=DAILY_TRAIN_OBS_CAP,
        min_obs=DAILY_MIN_OBS,
        refit_every=DAILY_REFIT_EVERY,
        show_progress=True,
        desc=f"{label} 주유소 직접모델",
    )

    fit["fair_retail_direct"] = fit["pred_actual_retail"]

    metrics = evaluate_prediction(
        fit,
        actual_col="actual_retail",
        pred_col="fair_retail_direct",
        exclude_col="is_excluded",
        label=f"{product}_retail_direct_huber",
    )
    metrics["x_cols"] = "|".join(x_cols)

    return fit, metrics


def fit_retail_two_layer_model(daily_frame_with_refinery: pd.DataFrame,
                               product: str) -> tuple[pd.DataFrame, pd.Series]:
    label = PRODUCT_MAP[product]["label"]

    df = daily_frame_with_refinery.copy()
    df["retail_spread_vs_fair_refinery"] = df["actual_retail"] - df["fair_refinery_pre"]

    x_cols = build_spread_x_cols(df)

    enough_refinery_pred = df["fair_refinery_pre"].notna().sum() >= DAILY_MIN_OBS

    if not enough_refinery_pred:
        metrics = pd.Series({
            "model_name": f"{product}_retail_two_layer_spread_huber",
            "clean_n": 0,
            "clean_mae": np.nan,
            "clean_median_ae": np.nan,
            "clean_bias": np.nan,
            "clean_abs_bias": np.nan,
            "clean_resid_acf1": np.nan,
            "clean_resid_acf7": np.nan,
            "x_cols": "|".join(x_cols),
            "status": "not_enough_refinery_prediction",
        })
        df["pred_retail_spread"] = np.nan
        df["fair_retail_two_layer"] = np.nan
        return df, metrics

    fit = rolling_huber_predict(
        df,
        target_col="retail_spread_vs_fair_refinery",
        x_cols=x_cols,
        exclude_col="is_excluded",
        train_obs_cap=DAILY_TRAIN_OBS_CAP,
        min_obs=DAILY_MIN_OBS,
        refit_every=DAILY_REFIT_EVERY,
        show_progress=True,
        desc=f"{label} 주유소 spread 모델",
    )

    fit["pred_retail_spread"] = fit["pred_retail_spread_vs_fair_refinery"]
    fit["fair_retail_two_layer"] = fit["fair_refinery_pre"] + fit["pred_retail_spread"]

    metrics = evaluate_prediction(
        fit,
        actual_col="actual_retail",
        pred_col="fair_retail_two_layer",
        exclude_col="is_excluded",
        label=f"{product}_retail_two_layer_spread_huber",
    )
    metrics["x_cols"] = "|".join(x_cols)
    metrics["status"] = "ok"

    return fit, metrics


def choose_final_retail_model(direct_metrics: pd.Series,
                              two_layer_metrics: pd.Series) -> str:
    direct_mae = direct_metrics.get("clean_mae", np.nan)
    two_mae = two_layer_metrics.get("clean_mae", np.nan)

    direct_n = direct_metrics.get("clean_n", 0)
    two_n = two_layer_metrics.get("clean_n", 0)

    direct_ok = np.isfinite(direct_mae) and int(direct_n) > 0
    two_ok = np.isfinite(two_mae) and int(two_n) > 0

    if direct_ok and two_ok:
        if two_mae <= direct_mae * TWO_LAYER_MAE_ALLOWANCE:
            return "two_layer"
        return "direct"

    if two_ok:
        return "two_layer"

    if direct_ok:
        return "direct"

    raise RuntimeError(
        "direct 모델과 two-layer 모델 모두 clean 평가에 실패했습니다. "
        "feature 결측, 제외기간, min_obs 설정을 확인하세요."
    )


def assert_prediction_enough(df: pd.DataFrame,
                             pred_col: str,
                             label: str,
                             min_count: int = 30):
    if pred_col not in df.columns:
        raise ValueError(f"[{label}] 예측 컬럼이 없습니다: {pred_col}")

    n = int(pd.to_numeric(df[pred_col], errors="coerce").notna().sum())

    if n < min_count:
        raise RuntimeError(
            f"[{label}] 예측값이 너무 적습니다. "
            f"{pred_col} non-null={n}, required>={min_count}"
        )

    print_log(f"[CHECK] {label}: {pred_col} non-null = {n:,}")

def build_final_output(
    product: str,
    meta: dict,
    daily_selected: pd.DataFrame,
    refinery_weekly_pred: pd.DataFrame | None,
    model_metrics: pd.DataFrame,
    band_metrics: pd.DataFrame,
    selected_model: str,
) -> pd.DataFrame:
    out = pd.DataFrame(index=daily_selected.index)

    # 주유소 실제가격 / 국제제품가격
    out["actual_gross_full"] = daily_selected["actual_retail"]
    out["international_full"] = daily_selected["intl_raw"]

    # 3단계에서 정책을 별도 적용하기 위해 pred_gross는 정책 미반영 적정가격으로 저장
    out["pred_gross"] = daily_selected["fair_retail_no_policy"]
    out["band_low"] = daily_selected["band_low"]
    out["band_high"] = daily_selected["band_high"]

    # 구버전 3단계 호환 방지용 placeholder
    # 새 3단계에서는 이 컬럼들을 세금 계산에 절대 쓰지 않는다.
    out["pred_pretax"] = np.nan
    out["tax_sum_full"] = np.nan

    # 모델 정보
    out["selected_model_name"] = selected_model
    out["selected_model_name_daily"] = "direct"
    out["selected_band_coverage"] = daily_selected["band_coverage"]

    # 판정/진단
    out["judge"] = daily_selected["judge"]
    out["inside_band"] = daily_selected["inside_band"]
    out["below_band"] = daily_selected["below_band"]
    out["above_band"] = daily_selected["above_band"]
    out["is_excluded"] = daily_selected["is_excluded"]

    # direct 모델 결과
    out["fair_retail_direct"] = daily_selected["fair_retail_direct"]

    # 정유사 관련 컬럼은 최종 주유소 결과에 넣지 않는다.
    # 정유사는 별도 파일로만 저장한다.
    out["retail_gap_원L"] = out["actual_gross_full"] - out["pred_gross"]

    out["source_retail_col"] = meta["retail_col"]
    out["source_intl_col"] = meta["intl_col"]

    out.index.name = DATE_COL
    return out.reset_index()


def save_yearly_plots(output_df: pd.DataFrame,
                         product: str,
                         output_dir: Path):
    label = PRODUCT_MAP[product]["label"]
    plot_dir = output_dir / f"{product}_yearly_plots"
    plot_dir.mkdir(parents=True, exist_ok=True)

    df = output_df.copy()
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    df = df.sort_values(DATE_COL)

    for year, ydf in df.groupby(df[DATE_COL].dt.year):
        if pd.isna(year):
            continue

        fig, ax = plt.subplots(figsize=(18, 7), constrained_layout=True)

        ax.plot(
            ydf[DATE_COL],
            ydf["actual_gross_full"],
            linewidth=1.4,
            label="국내 주유소 가격",
        )

        valid_band = ydf["band_low"].notna() & ydf["band_high"].notna()
        if valid_band.any():
            ax.fill_between(
                ydf.loc[valid_band, DATE_COL],
                ydf.loc[valid_band, "band_low"],
                ydf.loc[valid_band, "band_high"],
                alpha=0.25,
                label="적정 범위",
            )

        ax.plot(
            ydf[DATE_COL],
            ydf["pred_gross"],
            linewidth=1.6,
            label="정책 미반영 적정가격",
        )

        if "fair_retail_direct" in ydf.columns and ydf["fair_retail_direct"].notna().any():
            ax.plot(
                ydf[DATE_COL],
                ydf["fair_retail_direct"],
                linewidth=1.0,
                linestyle="--",
                alpha=0.7,
                label="직접모델 참고선",
            )

        ax.set_title(f"{int(year)}년 {label} 2단계-v2: 정책 미반영 적정가격")
        ax.set_xlabel("date")
        ax.set_ylabel("원/리터")
        ax.grid(True, alpha=0.3)
        ax.legend()

        out_file = plot_dir / f"{product}_{int(year)}_step2.png"
        fig.savefig(out_file, dpi=150, bbox_inches="tight")
        plt.close(fig)

    print_log(f"연도별 그래프 저장 완료: {plot_dir}")


def run_step2(product: str = PRODUCT):
    print_log(f"실행 시작: {product}")

    if not RAW_INPUT_FILE.exists():
        raise FileNotFoundError(f"원본 CSV 없음: {RAW_INPUT_FILE}")

    raw_df = pd.read_csv(RAW_INPUT_FILE)

    # -----------------------------------------------------
    # 1) 주유소 IRF만 주유소 모델에 사용
    # -----------------------------------------------------
    consumer_summary, consumer_weights = load_stage1_irf(
        product,
        layer="consumer",
    )

    daily_frame, meta = build_daily_retail_frame(
        raw_df,
        product,
        consumer_weights,
    )

    print_log(f"유종: {meta['label']}")
    print_log(f"소매가격 컬럼: {meta['retail_col']}")
    print_log(f"국제제품가격 컬럼: {meta['intl_col']}")
    print_log(f"일별 주유소 frame: {len(daily_frame):,}행")
    print_log("최종 주유소 모델에는 정유사 데이터를 사용하지 않습니다.")

    # -----------------------------------------------------
    # 2) 주유소 direct 모델
    # -----------------------------------------------------
    retail_direct, direct_metrics = fit_retail_direct_model(
        daily_frame,
        product,
    )

    assert_prediction_enough(
        retail_direct,
        "fair_retail_direct",
        f"{product} 주유소 direct",
        min_count=180,
    )

    combined = daily_frame.copy()
    combined["fair_retail_direct"] = retail_direct["fair_retail_direct"]

    # 최종 주유소 적정가격은 direct 모델로 고정
    selected_model = "direct"
    combined["fair_retail_no_policy"] = combined["fair_retail_direct"]
    combined["selected_model_name_daily"] = "direct"

    print_log("최종 사용 모델: direct")
    print_log("정유사 two-layer 모델은 사용하지 않습니다.")

    # -----------------------------------------------------
    # 3) 최종 band
    # -----------------------------------------------------
    banded, band_metrics = select_best_band(
        combined,
        actual_col="actual_retail",
        pred_col="fair_retail_no_policy",
        exclude_col="is_excluded",
    )

    combined["band_low"] = banded["band_low"]
    combined["band_high"] = banded["band_high"]
    combined["band_hist_n"] = banded["band_hist_n"]
    combined["band_coverage"] = banded["band_coverage"]
    combined["inside_band"] = banded["inside_band"]
    combined["below_band"] = banded["below_band"]
    combined["above_band"] = banded["above_band"]
    combined["judge"] = banded["judge"]

    # -----------------------------------------------------
    # 4) 주유소 모델 metrics
    # -----------------------------------------------------
    final_metrics = evaluate_prediction(
        combined,
        actual_col="actual_retail",
        pred_col="fair_retail_no_policy",
        exclude_col="is_excluded",
        label=f"{product}_selected_direct",
    )

    model_metrics_rows = [
        direct_metrics,
        final_metrics,
    ]

    # -----------------------------------------------------
    # 5) 정유사 모델은 별도 산출물로만 실행
    # -----------------------------------------------------
    refinery_weekly_pred = pd.DataFrame()
    weekly_frame = pd.DataFrame()

    if RUN_REFINERY_DIAGNOSTIC:
        try:
            refinery_summary, refinery_weights = load_stage1_irf(
                product,
                layer="refinery",
            )

            weekly_frame = build_weekly_refinery_frame(
                daily_frame,
                refinery_weights=refinery_weights,
            )

            print_log(f"정유사 별도 weekly frame: {len(weekly_frame):,}행")

            if len(weekly_frame) > 0:
                refinery_weekly_pred, refinery_metrics = fit_refinery_weekly_model(
                    weekly_frame,
                    product,
                )

                assert_prediction_enough(
                    refinery_weekly_pred,
                    "fair_refinery_pre",
                    f"{product} 정유사 weekly diagnostic",
                    min_count=30,
                )

                model_metrics_rows.append(refinery_metrics)

        except Exception as e:
            print_log(f"[WARN] 정유사 별도 진단 모델 실행 실패: {e}")
            refinery_weekly_pred = pd.DataFrame()
            weekly_frame = pd.DataFrame()

    model_metrics = pd.DataFrame(model_metrics_rows)

    # -----------------------------------------------------
    # 6) 최종 output
    # -----------------------------------------------------
    output_df = build_final_output(
        product=product,
        meta=meta,
        daily_selected=combined,
        refinery_weekly_pred=refinery_weekly_pred,
        model_metrics=model_metrics,
        band_metrics=band_metrics,
        selected_model=selected_model,
    )

    # -----------------------------------------------------
    # 7) 저장
    # -----------------------------------------------------
    product_dir = OUTPUT_DIR / product
    product_dir.mkdir(parents=True, exist_ok=True)

    daily_frame.reset_index().to_csv(
        product_dir / f"{product}_step2_daily_frame.csv",
        index=False,
        encoding="utf-8-sig",
    )

    combined.reset_index().to_csv(
        product_dir / f"{product}_retail_model_internal_predictions.csv",
        index=False,
        encoding="utf-8-sig",
    )

    if len(weekly_frame) > 0:
        weekly_frame.reset_index().to_csv(
            product_dir / f"{product}_step2_weekly_refinery_frame.csv",
            index=False,
            encoding="utf-8-sig",
        )

    if len(refinery_weekly_pred) > 0:
        refinery_weekly_pred.reset_index().to_csv(
            product_dir / f"{product}_refinery_weekly_predictions.csv",
            index=False,
            encoding="utf-8-sig",
        )

    model_metrics.to_csv(
        product_dir / f"{product}_model_comparison.csv",
        index=False,
        encoding="utf-8-sig",
    )

    band_metrics.to_csv(
        product_dir / f"{product}_band_comparison.csv",
        index=False,
        encoding="utf-8-sig",
    )

    output_df.to_csv(
        OUTPUT_DIR / f"{product}_production_predictions_full_calendar.csv",
        index=False,
        encoding="utf-8-sig",
    )

    save_yearly_plots(output_df, product, OUTPUT_DIR)

    print_log(f"완료: {product}")
    print_log(f"최종 파일: {OUTPUT_DIR / f'{product}_production_predictions_full_calendar.csv'}")

    return {
        "product": product,
        "meta": meta,
        "daily_frame": daily_frame,
        "weekly_frame": weekly_frame,
        "refinery_weekly_pred": refinery_weekly_pred,
        "combined": combined,
        "output_df": output_df,
        "model_metrics": model_metrics,
        "band_metrics": band_metrics,
        "selected_model": selected_model,
    }


print("[READY] 2단계-v2 실행 함수 정의 완료")

[READY] 2단계-v2 실행 함수 정의 완료


## 휘발유 / 경유 실행


In [ ]:
# =========================================================
# 5. 휘발유 / 경유 실행
# =========================================================

gasoline = run_step2("gasoline")

print("\n[휘발유 model metrics]")
display(gasoline["model_metrics"]) if display is not None else print(gasoline["model_metrics"])

print("\n[휘발유 band metrics]")
display(gasoline["band_metrics"]) if display is not None else print(gasoline["band_metrics"])


diesel = run_step2("diesel")

print("\n[경유 model metrics]")
display(diesel["model_metrics"]) if display is not None else print(diesel["model_metrics"])

print("\n[경유 band metrics]")
display(diesel["band_metrics"]) if display is not None else print(diesel["band_metrics"])

[STEP2-V2] 실행 시작: gasoline
[STEP2-V2] [IRF LOAD] product=gasoline, layer=consumer, unit=days, min_lag=1
[STEP2-V2] [IRF LOAD] source=/content/drive/MyDrive/Data_analysis/The appropriateness of domestic oil prices compared to international oil prices/산업부/시차분석/gasoline_lag_analysis
[STEP2-V2] 유종: 휘발유
[STEP2-V2] 소매가격 컬럼: 보통휘발유_평균
[STEP2-V2] 국제제품가격 컬럼: 휘발유92RON_원리터
[STEP2-V2] 일별 주유소 frame: 6,630행
[STEP2-V2] 최종 주유소 모델에는 정유사 데이터를 사용하지 않습니다.


휘발유 주유소 직접모델:   0%|          | 0/6630 [00:00<?, ?it/s]

[STEP2-V2] [CHECK] gasoline 주유소 direct: fair_retail_direct non-null = 6,099
[STEP2-V2] 최종 사용 모델: direct
[STEP2-V2] 정유사 two-layer 모델은 사용하지 않습니다.
[STEP2-V2] [IRF LOAD] product=gasoline, layer=refinery, unit=weeks, min_lag=0
[STEP2-V2] [IRF LOAD] source=/content/drive/MyDrive/Data_analysis/The appropriateness of domestic oil prices compared to international oil prices/산업부/시차분석/gasoline_refinery_weekly_lag_analysis
[STEP2-V2] 정유사 별도 weekly frame: 943행


휘발유 정유사 주간모델:   0%|          | 0/943 [00:00<?, ?it/s]

[STEP2-V2] [CHECK] gasoline 정유사 weekly diagnostic: fair_refinery_pre non-null = 824


...

경유 주유소 직접모델:   0%|          | 0/6630 [00:00<?, ?it/s]

[STEP2-V2] [CHECK] diesel 주유소 direct: fair_retail_direct non-null = 4,687
[STEP2-V2] 최종 사용 모델: direct
[STEP2-V2] 정유사 two-layer 모델은 사용하지 않습니다.
[STEP2-V2] [IRF LOAD] product=diesel, layer=refinery, unit=weeks, min_lag=0
[STEP2-V2] [IRF LOAD] source=/content/drive/MyDrive/Data_analysis/The appropriateness of domestic oil prices compared to international oil prices/산업부/시차분석/diesel_refinery_weekly_lag_analysis
[STEP2-V2] 정유사 별도 weekly frame: 704행


경유 정유사 주간모델:   0%|          | 0/704 [00:00<?, ?it/s]

[STEP2-V2] [CHECK] diesel 정유사 weekly diagnostic: fair_refinery_pre non-null = 621


/tmp/ipykernel_742/191039492.py:304: UserWarning: Glyph 50896 (\N{HANGUL SYLLABLE WEON}) missing from font(s) DejaVu Sans.
  fig.savefig(out_file, dpi=150, bbox_inches="tight")
/tmp/ipykernel_742/191039492.py:304: UserWarning: Glyph 47532 (\N{HANGUL SYLLABLE RI}) missing from font(s) DejaVu Sans.
  fig.savefig(out_file, dpi=150, bbox_inches="tight")
/tmp/ipykernel_742/191039492.py:304: UserWarning: Glyph 53552 (\N{HANGUL SYLLABLE TEO}) missing from font(s) DejaVu Sans.
  fig.savefig(out_file, dpi=150, bbox_inches="tight")
/tmp/ipykernel_742/191039492.py:304: UserWarning: Glyph 45380 (\N{HANGUL SYLLABLE NYEON}) missing from font(s) DejaVu Sans.
  fig.savefig(out_file, dpi=150, bbox_inches="tight")
/tmp/ipykernel_742/191039492.py:304: UserWarning: Glyph 44221 (\N{HANGUL SYLLABLE GYEONG}) missing from font(s) DejaVu Sans.
  fig.savefig(out_file, dpi=150, bbox_inches="tight")
/tmp/ipykernel_742/191039492.py:304: UserWarning: Glyph 50976 (\N{HANGUL SYLLABLE YU}) missing from font(s) DejaVu 